# LeetCode 3636 — Threshold Majority Queries bằng Python

- Link: https://leetcode.com/problems/threshold-majority-queries/
- Độ khó: **Hard**
- Chủ đề: **Mode của đoạn** (range mode query), **thuật toán Mo có rollback** (chỉ thêm, không xoá), chia khối tiền tính, nén giá trị

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def subarrayMajority(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra câu trả lời chỉ phụ thuộc **mode** của đoạn (tần suất cao nhất, hoà thì giá trị nhỏ nhất): nếu tần suất đó `≥ threshold` thì trả mode, ngược lại `-1`.
2. Hiểu vì sao Mo **thông thường** khó (xoá phần tử làm "max tần suất" khó cập nhật) và cách **Mo rollback** tránh xoá.
3. Cài **chia khối tiền tính**: mode của `[đầu khối b, r]` cho mọi `b, r`, rồi bổ sung phần lẻ bên trái bằng danh sách vị trí + `bisect`.
4. Nén giá trị để dùng mảng đếm thay cho từ điển.
5. Kiểm chứng bằng `Counter` trực tiếp.

## 2. Đề bài & ví dụ

Với truy vấn `[l, r, threshold]`, trả phần tử của `nums[l..r]` xuất hiện **ít nhất** `threshold` lần, chọn phần tử có tần suất **cao nhất** (hoà ⇒ giá trị **nhỏ nhất**); không có ⇒ `-1`.

**Ví dụ 1:** `nums = [1,1,2,2,1,1]`, `queries = [[0,5,4],[0,3,3],[2,3,2]]` → `[1, -1, 2]`.

**Ví dụ 2:** `nums = [3,2,3,2,3,2,3]`, `queries = [[0,6,4],[1,5,2],[2,4,1],[3,3,1]]` → `[3, 2, 3, 2]`.

**Ràng buộc:** `n ≤ 10^4`, `q ≤ 5·10^4`.

In [1]:
from typing import List
from collections import Counter, defaultdict
from bisect import bisect_left, bisect_right
import random, time

examples = [
    ([1, 1, 2, 2, 1, 1], [[0, 5, 4], [0, 3, 3], [2, 3, 2]], [1, -1, 2]),
    ([3, 2, 3, 2, 3, 2, 3], [[0, 6, 4], [1, 5, 2], [2, 4, 1], [3, 3, 1]], [3, 2, 3, 2]),
]
for a, qs, exp in examples:
    print(a, qs, "-> kỳ vọng", exp)

[1, 1, 2, 2, 1, 1] [[0, 5, 4], [0, 3, 3], [2, 3, 2]] -> kỳ vọng [1, -1, 2]
[3, 2, 3, 2, 3, 2, 3] [[0, 6, 4], [1, 5, 2], [2, 4, 1], [3, 3, 1]] -> kỳ vọng [3, 2, 3, 2]


## 3. Kiến thức nền

### 3.1. Rút gọn về mode

Phần tử được chọn phải có tần suất cao nhất trong đoạn (với tie-break nhỏ nhất) **và** `≥ threshold`. Nếu mode có tần suất `< threshold` thì không phần tử nào đạt ⇒ `-1`. Vậy chỉ cần tính mode của đoạn.

In [2]:
def mode(seg):
    c = Counter(seg)
    return min(c.items(), key=lambda kv: (-kv[1], kv[0]))       # (giá trị, tần suất)

print(mode([1, 1, 2, 2]), mode([3, 2, 3, 2, 3]))

(1, 2) (3, 3)


### 3.2. Thuật toán Mo và vấn đề khi xoá

Mo sắp truy vấn theo `(khối của l, r)` và dịch hai con trỏ, mỗi bước **thêm** hoặc **xoá** một phần tử. Thêm: cập nhật `max` dễ. Xoá: nếu phần tử bị xoá là mode duy nhất, cần biết mode mới — khó trong `O(1)` khi có tie-break theo giá trị.

**Mo rollback:** với các truy vấn có `l` cùng khối, `r` tăng dần ⇒ con trỏ phải chỉ **thêm**. Phần bên trái (từ `l` tới cuối khối) được thêm tạm thời rồi **hoàn tác** (rollback) — không cần cập nhật `max` khi hoàn tác vì ta lưu lại giá trị cũ.

### 3.3. Chia khối tiền tính mode

Với khối kích thước `B`, tính `tab[b][r]` = mode của `[b·B, r]` cho mọi `b`, `r ≥ b·B` (quét thêm dần, `O(n²/B)`). Truy vấn `[l, r]`: lấy `b` = khối đầu tiên bắt đầu `≥ l`; mode thật hoặc là `tab[b][r]`, hoặc là một giá trị xuất hiện trong phần `[l, b·B)` (≤ `B` phần tử) — kiểm tra tần suất thật của mỗi ứng viên bằng danh sách vị trí + `bisect`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mode của `[l, r]` = mode của phần "lõi" lớn `[l', r]` **hoặc** một phần tử nằm trong phần thêm `[l, l')`: thêm phần tử chỉ có thể làm tăng tần suất của chính các giá trị được thêm. Đây là lý do cả hai kỹ thuật (Mo rollback và chia khối) chỉ cần xét thêm các giá trị ở phần lẻ.

Ví dụ 1, truy vấn `[0,3,3]`: đoạn `[1,1,2,2]`, mode `1` (tần suất 2, hoà với `2`, chọn nhỏ hơn) nhưng `2 < 3` ⇒ `-1`. ✔

## 5. Cách 1 — Brute Force: `Counter` mỗi truy vấn

In [3]:
def brute(nums, queries):
    out = []
    for l, r, t in queries:
        v, f = mode(nums[l:r + 1])
        out.append(v if f >= t else -1)
    return out

for a, qs, exp in examples:
    print(brute(a, qs), "kỳ vọng", exp)

[1, -1, 2] kỳ vọng [1, -1, 2]
[3, 2, 3, 2] kỳ vọng [3, 2, 3, 2]


**Nút thắt:** `O(n · q) = 5·10^8` trường hợp xấu.

## 6. Ý tưởng thuật toán / bước đột phá

**Mo rollback** (lời giải nộp), với `B ≈ n / √q`:

1. Nén giá trị về `0..m−1` (thứ tự tăng ⇒ "giá trị nhỏ nhất" ⇔ "hạng nhỏ nhất").
2. Nhóm truy vấn theo khối của `l`; trong mỗi khối sắp theo `r`.
3. Truy vấn nằm gọn trong một khối: đếm trực tiếp.
4. Ngược lại: con trỏ phải `cr` bắt đầu ở cuối khối, chỉ tăng; duy trì `(bf, bv)` = mode của `[cuối khối + 1, cr]`. Với mỗi truy vấn, thêm tạm `[l, cuối khối]`, đọc mode, rồi trừ lại số đếm.

## 7. Các cách tiếp theo

- **Cách 2:** chia khối tiền tính `tab[b][r]` + ứng viên phần lẻ + `bisect`.
- **Cách 3:** Mo rollback (lời giải nộp).
- **Cách 4:** "ngây thơ có nén": `Counter` → mảng đếm trên giá trị đã nén (vẫn `O(nq)` nhưng hằng số nhỏ) — để thấy giới hạn của tối ưu hằng số.

### 7.1. Cách 2 — Chia khối tiền tính

In [4]:
def block_table(nums, queries):
    n = len(nums); vals = sorted(set(nums)); rk = {v: i for i, v in enumerate(vals)}; a = [rk[x] for x in nums]; m = len(vals)
    B = max(1, int(n ** 0.5)); nb = (n + B - 1) // B
    tab = []
    for b in range(nb):
        cnt = [0] * m; row = [None] * n; bf = bv = 0
        for r in range(b * B, n):
            v = a[r]; cnt[v] += 1; c = cnt[v]
            if c > bf or (c == bf and v < bv): bf, bv = c, v
            row[r] = (bf, bv)
        tab.append(row)
    pos = defaultdict(list)
    for i, v in enumerate(a): pos[v].append(i)
    out = []
    for l, r, t in queries:
        bs = (l + B - 1) // B
        if bs >= nb or bs * B > r:
            c = Counter(a[l:r + 1]); bf, bv = min(((cc, v) for v, cc in c.items()), key=lambda x: (-x[0], x[1]))
        else:
            bf, bv = tab[bs][r]
            for v in set(a[l:bs * B]):
                p = pos[v]; c = bisect_right(p, r) - bisect_left(p, l)
                if c > bf or (c == bf and v < bv): bf, bv = c, v
        out.append(vals[bv] if bf >= t else -1)
    return out

for a, qs, exp in examples:
    print(block_table(a, qs), "kỳ vọng", exp)

[1, -1, 2] kỳ vọng [1, -1, 2]
[3, 2, 3, 2] kỳ vọng [3, 2, 3, 2]


### 7.2. Cách 3 — Mo rollback (lời giải chính)

In [5]:
def mo_rollback(nums, queries):
    n = len(nums); vals = sorted(set(nums)); rk = {v: i for i, v in enumerate(vals)}; a = [rk[x] for x in nums]; m = len(vals)
    B = max(1, int(n / max(1, len(queries)) ** 0.5))
    out = [0] * len(queries); cnt = [0] * m
    groups = defaultdict(list)
    for i, (l, r, t) in enumerate(queries): groups[l // B].append(i)
    for b, ids in groups.items():
        ids.sort(key=lambda i: queries[i][1])
        bend = (b + 1) * B                                   # vị trí đầu khối kế tiếp
        cr = bend - 1; bf = bv = 0; touched = []
        for i in ids:
            l, r, t = queries[i]
            if r < bend:                                     # gọn trong một khối
                c = Counter(a[l:r + 1]); f, v = min(((cc, vv) for vv, cc in c.items()), key=lambda x: (-x[0], x[1]))
                out[i] = vals[v] if f >= t else -1; continue
            while cr < r:                                    # chỉ thêm bên phải
                cr += 1; v = a[cr]; cnt[v] += 1; touched.append(v); c = cnt[v]
                if c > bf or (c == bf and v < bv): bf, bv = c, v
            f0, v0 = bf, bv
            for j in range(l, bend):                         # thêm tạm bên trái
                v = a[j]; cnt[v] += 1; c = cnt[v]
                if c > f0 or (c == f0 and v < v0): f0, v0 = c, v
            for j in range(l, bend): cnt[a[j]] -= 1          # hoàn tác
            out[i] = vals[v0] if f0 >= t else -1
        for v in touched: cnt[v] = 0
    return out

for a, qs, exp in examples:
    print(mo_rollback(a, qs), "kỳ vọng", exp)

[1, -1, 2] kỳ vọng [1, -1, 2]
[3, 2, 3, 2] kỳ vọng [3, 2, 3, 2]


### 7.3. Cách 4 — Đếm trực tiếp trên giá trị đã nén

In [6]:
def naive_compressed(nums, queries):
    vals = sorted(set(nums)); rk = {v: i for i, v in enumerate(vals)}; a = [rk[x] for x in nums]; m = len(vals)
    cnt = [0] * m; out = []
    for l, r, t in queries:
        bf = bv = 0
        for j in range(l, r + 1):
            v = a[j]; cnt[v] += 1; c = cnt[v]
            if c > bf or (c == bf and v < bv): bf, bv = c, v
        for j in range(l, r + 1): cnt[a[j]] = 0
        out.append(vals[bv] if bf >= t else -1)
    return out

for a, qs, exp in examples:
    print(naive_compressed(a, qs), "kỳ vọng", exp)

[1, -1, 2] kỳ vọng [1, -1, 2]
[3, 2, 3, 2] kỳ vọng [3, 2, 3, 2]


## 8. TRACE từng bước

In [7]:
def trace_rollback(nums, queries, B):
    vals = sorted(set(nums)); rk = {v: i for i, v in enumerate(vals)}; a = [rk[x] for x in nums]
    print("giá trị nén:", vals, "| a =", a, "| B =", B)
    groups = defaultdict(list)
    for i, (l, r, t) in enumerate(queries): groups[l // B].append(i)
    for b in sorted(groups):
        ids = sorted(groups[b], key=lambda i: queries[i][1]); bend = (b + 1) * B; cnt = Counter(); cr = bend - 1; bf = bv = 0
        print(f"khối {b} (kết thúc trước {bend}): truy vấn theo r tăng {[(queries[i]) for i in ids]}")
        for i in ids:
            l, r, t = queries[i]
            if r < bend:
                v, f = mode(a[l:r + 1]); print(f"   {queries[i]}: gọn trong khối, mode {vals[v]} tần suất {f}"); continue
            while cr < r:
                cr += 1; v = a[cr]; cnt[v] += 1
                if cnt[v] > bf or (cnt[v] == bf and v < bv): bf, bv = cnt[v], v
            tmp = Counter(cnt); f0, v0 = bf, bv
            for j in range(l, bend):
                tmp[a[j]] += 1
                if tmp[a[j]] > f0 or (tmp[a[j]] == f0 and a[j] < v0): f0, v0 = tmp[a[j]], a[j]
            print(f"   {queries[i]}: phải tới {cr}: mode phần phải ({vals[bv]}, {bf}); thêm tạm [{l},{bend - 1}] -> ({vals[v0]}, {f0}) -> {'%d' % vals[v0] if f0 >= t else -1}")

trace_rollback([3, 2, 3, 2, 3, 2, 3], [[0, 6, 4], [1, 5, 2], [2, 4, 1], [3, 3, 1]], B=2)

giá trị nén: [2, 3] | a = [1, 0, 1, 0, 1, 0, 1] | B = 2
khối 0 (kết thúc trước 2): truy vấn theo r tăng [[1, 5, 2], [0, 6, 4]]
   [1, 5, 2]: phải tới 5: mode phần phải (2, 2); thêm tạm [1,1] -> (2, 3) -> 2
   [0, 6, 4]: phải tới 6: mode phần phải (3, 3); thêm tạm [0,1] -> (3, 4) -> 3
khối 1 (kết thúc trước 4): truy vấn theo r tăng [[3, 3, 1], [2, 4, 1]]
   [3, 3, 1]: gọn trong khối, mode 2 tần suất 1
   [2, 4, 1]: phải tới 4: mode phần phải (3, 1); thêm tạm [2,3] -> (3, 2) -> 3


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def subarrayMajority(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        n = len(nums)
        vals = sorted(set(nums))
        rk = {v: i for i, v in enumerate(vals)}
        a = [rk[x] for x in nums]
        m = len(vals)
        B = max(1, int(n / max(1, len(queries)) ** 0.5))
        out = [0] * len(queries)
        cnt = [0] * m
        groups = {}
        for i, (l, r, t) in enumerate(queries):
            groups.setdefault(l // B, []).append(i)
        for b, ids in groups.items():
            ids.sort(key=lambda i: queries[i][1])
            bend = (b + 1) * B
            cr = bend - 1
            bf = bv = 0
            touched = []
            for i in ids:
                l, r, t = queries[i]
                if r < bend:
                    f0 = v0 = 0
                    for j in range(l, r + 1):
                        v = a[j]; cnt[v] += 1; c = cnt[v]
                        if c > f0 or (c == f0 and v < v0):
                            f0, v0 = c, v
                    for j in range(l, r + 1):
                        cnt[a[j]] -= 1
                    out[i] = vals[v0] if f0 >= t else -1
                    continue
                while cr < r:
                    cr += 1; v = a[cr]; cnt[v] += 1; touched.append(v); c = cnt[v]
                    if c > bf or (c == bf and v < bv):
                        bf, bv = c, v
                f0, v0 = bf, bv
                for j in range(l, bend):
                    v = a[j]; cnt[v] += 1; c = cnt[v]
                    if c > f0 or (c == f0 and v < v0):
                        f0, v0 = c, v
                for j in range(l, bend):
                    cnt[a[j]] -= 1
                out[i] = vals[v0] if f0 >= t else -1
            for v in touched:
                cnt[v] = 0
        return out

sol = Solution()
for a, qs, exp in examples:
    got = sol.subarrayMajority(a, qs)
    print(got, "✔" if got == exp else "✘")

[1, -1, 2] ✔
[3, 2, 3, 2] ✔


## 10. Vì sao thuật toán đúng

1. **Rút gọn:** đáp án = mode (theo thứ tự `(−tần suất, giá trị)`) nếu tần suất `≥ threshold`, ngược lại `-1` (3.1).
2. **Nén giá trị** giữ thứ tự ⇒ so sánh hạng tương đương so sánh giá trị.
3. **Mo rollback:** với truy vấn trong khối `b` có `r ≥ bend`, sau vòng `while`, `cnt` chứa đúng tần suất của `[bend, r]` và `(bf, bv)` là mode của đoạn đó (chỉ thêm ⇒ cập nhật max đúng). Thêm tạm `[l, bend)`: tần suất của mọi giá trị trở thành tần suất trong `[l, r]`; mode mới là `max` giữa mode cũ và các giá trị vừa được tăng (mọi giá trị khác giữ nguyên tần suất) ⇒ `(f0, v0)` đúng. Hoàn tác đưa `cnt` về trạng thái `[bend, r]`. Các truy vấn được sắp theo `r` nên `cr` chỉ tăng.
4. **Chia khối:** mode của `[l, r]` thuộc `{mode của [bs·B, r]} ∪ {giá trị trong [l, bs·B)}` (lập luận như 3); tần suất thật của mỗi ứng viên lấy bằng hai `bisect`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. `Counter` | `O(n q)` | `O(n)` |
| 2. Chia khối tiền tính | `O(n²/B + q·B·log n)` ≈ `O(n√n + q√n log n)` | `O(n²/B)` |
| 3. Mo rollback | `O(n · n/B + q · B)` ≈ `O(n √q)` với `B = n/√q` | `O(n)` |
| 4. Đếm nén | `O(n q)` (hằng số nhỏ) | `O(n)` |

## 12. Bộ test

In [9]:
METHODS = [brute, block_table, mo_rollback, naive_compressed, lambda a, q: Solution().subarrayMajority(a, q)]
for a, qs, exp in examples:
    assert all(f(a, qs) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [10]:
# 12.2 ca biên
assert Solution().subarrayMajority([5], [[0, 0, 1]]) == [5]
assert Solution().subarrayMajority([9, 8, 7], [[0, 2, 1], [0, 2, 2]]) == [7, -1]       # hoà: chọn nhỏ nhất
assert Solution().subarrayMajority([10**9] * 4, [[0, 3, 4], [1, 2, 3]]) == [10**9, -1]
print("Ca biên: OK")

Ca biên: OK


In [11]:
# 12.3 ngẫu nhiên
rng = random.Random(3636)
for _ in range(500):
    n = rng.randint(1, 40); a = [rng.randint(1, 6) for _ in range(n)]; qs = []
    for _ in range(rng.randint(1, 30)):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, rng.randint(1, r - l + 1)])
    b = brute(a, qs)
    assert all(f(a, qs) == b for f in METHODS[1:]), (a, qs)
print("500 test ngẫu nhiên: OK")

500 test ngẫu nhiên: OK


In [12]:
# 12.4 test lớn: n = 10^4, q = 5·10^4
rng = random.Random(1)
n = 10**4
a = [rng.randint(1, 10**9) if rng.random() < 0.5 else rng.randint(1, 50) for _ in range(n)]; qs = []
for _ in range(5 * 10**4):
    l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, rng.randint(1, max(1, (r - l + 1) // 10))])
t0 = time.time(); x = Solution().subarrayMajority(a, qs); t1 = time.time()
y = block_table(a, qs); t2 = time.time()
print(f"Mo rollback: {t1 - t0:.2f}s | chia khối: {t2 - t1:.2f}s | khớp: {x == y} | số câu trả lời -1: {x.count(-1)}")

Mo rollback: 0.37s | chia khối: 1.00s | khớp: True | số câu trả lời -1: 39984


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chọn phần tử đầu tiên đạt ngưỡng (không phải tần suất cao nhất)

In [13]:
def wrong_first_reaching(nums, queries):
    out = []
    for l, r, t in queries:
        c = Counter(nums[l:r + 1]); ok = sorted(v for v, f in c.items() if f >= t)
        out.append(ok[0] if ok else -1)                                   # SAI: nhỏ nhất đạt ngưỡng
    return out

a, qs = [5, 1, 5, 5, 1], [[0, 4, 2]]
print("sai:", wrong_first_reaching(a, qs), "| đúng:", brute(a, qs))

sai: [1] | đúng: [5]


### Lỗi 2 — Tie-break chọn giá trị **lớn** nhất

In [14]:
def wrong_tie(nums, queries):
    out = []
    for l, r, t in queries:
        c = Counter(nums[l:r + 1]); v, f = max(c.items(), key=lambda kv: (kv[1], kv[0]))   # SAI
        out.append(v if f >= t else -1)
    return out

print("sai:", wrong_tie([9, 8, 7], [[0, 2, 1]]), "| đúng:", brute([9, 8, 7], [[0, 2, 1]]))

sai: [9] | đúng: [7]


### Lỗi 3 — Mo rollback quên hoàn tác phần bên trái

In [15]:
def wrong_no_rollback(nums, queries):
    vals = sorted(set(nums)); rk = {v: i for i, v in enumerate(vals)}; a = [rk[x] for x in nums]; m = len(vals)
    B = 2; out = [0] * len(queries); cnt = [0] * m; groups = defaultdict(list)
    for i, (l, r, t) in enumerate(queries): groups[l // B].append(i)
    for b, ids in groups.items():
        ids.sort(key=lambda i: queries[i][1]); bend = (b + 1) * B; cr = bend - 1; bf = bv = 0
        for i in ids:
            l, r, t = queries[i]
            if r < bend:
                v, f = mode(a[l:r + 1]); out[i] = vals[v] if f >= t else -1; continue
            while cr < r:
                cr += 1; v = a[cr]; cnt[v] += 1
                if cnt[v] > bf or (cnt[v] == bf and v < bv): bf, bv = cnt[v], v
            f0, v0 = bf, bv
            for j in range(l, bend):
                v = a[j]; cnt[v] += 1
                if cnt[v] > f0 or (cnt[v] == f0 and v < v0): f0, v0 = cnt[v], v
            out[i] = vals[v0] if f0 >= t else -1                                # SAI: không trừ lại
        cnt = [0] * m
    return out

rng = random.Random(17)
while True:
    n = rng.randint(4, 10); a = [rng.randint(1, 3) for _ in range(n)]; qs = []
    for _ in range(4):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, 1])
    if wrong_no_rollback(a, qs) != brute(a, qs): break
print("phản ví dụ:", a, qs, "| sai:", wrong_no_rollback(a, qs), "| đúng:", brute(a, qs))

phản ví dụ: [3, 2, 2, 1, 1, 2, 1, 2] [[4, 7, 1], [0, 3, 1], [4, 6, 1], [1, 6, 1]] | sai: [1, 2, 1, 2] | đúng: [1, 2, 1, 1]


### Lỗi 4 — So sánh giá trị gốc nhưng lưu hạng (hoặc ngược lại)

Khi đã nén, mọi so sánh tie-break phải dùng **hạng**; khi trả lời phải đổi về giá trị gốc `vals[hạng]`. Trộn hai hệ làm sai với giá trị lớn.

In [16]:
vals_demo = sorted({10**9, 7}); print("hạng của 7:", vals_demo.index(7), "| hạng của 10^9:", vals_demo.index(10**9), "| trả về phải là vals[hạng]")

hạng của 7: 0 | hạng của 10^9: 1 | trả về phải là vals[hạng]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phần tử xuất hiện nhiều nhất trong đoạn" ⇒ range mode — không có cấu trúc đẹp như sum/min ⇒ **Mo** hoặc **chia khối**.
- Mo mà phép xoá khó ⇒ **Mo rollback** (chỉ thêm, hoàn tác phần trái).
- Tie-break theo giá trị ⇒ nén giá trị theo thứ tự tăng để so sánh hạng.
- `n` nhỏ (`10^4`) cho phép tiền tính `O(n√n)` bộ nhớ.

## 15. Biến thể của bài

1. **Phần tử chiếm quá nửa** (majority > 1/2): Boyer–Moore trên segment tree + kiểm tra tần suất bằng `bisect` — `O(log n)` mỗi truy vấn.
2. **Đếm số giá trị phân biệt trong đoạn**: Mo thường (xoá dễ) hoặc BIT offline.
3. **Mode với cập nhật điểm**: Mo có thời gian (Mo with updates).
4. **Tần suất cao nhất (không cần giá trị)**: Mo thường + mảng "đếm số giá trị có tần suất f".

In [17]:
# Biến thể 1: majority > 1/2 bằng segment tree Boyer–Moore
def majority_queries(nums, queries):
    n = len(nums); size = 1
    while size < n: size *= 2
    t = [(0, 0)] * (2 * size)
    def merge(x, y):
        (a, ca), (b, cb) = x, y
        if a == b: return (a, ca + cb)
        return (a, ca - cb) if ca >= cb else (b, cb - ca)
    for i, v in enumerate(nums): t[size + i] = (v, 1)
    for i in range(size - 1, 0, -1): t[i] = merge(t[2 * i], t[2 * i + 1])
    pos = defaultdict(list)
    for i, v in enumerate(nums): pos[v].append(i)
    out = []
    for l, r in queries:
        res = (0, 0); lo, hi = l + size, r + size + 1; left = []; right = []
        while lo < hi:
            if lo & 1: left.append(t[lo]); lo += 1
            if hi & 1: hi -= 1; right.append(t[hi])
            lo //= 2; hi //= 2
        for x in left + right[::-1]: res = merge(res, x)
        cand = res[0]; p = pos.get(cand, [])
        out.append(cand if 2 * (bisect_right(p, r) - bisect_left(p, l)) > r - l + 1 else -1)
    return out

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 20); a = [rng.randint(1, 3) for _ in range(n)]; qs = []
    for _ in range(10):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    exp = []
    for l, r in qs:
        v, f = mode(a[l:r + 1]); exp.append(v if 2 * f > r - l + 1 else -1)
    assert majority_queries(a, qs) == exp
print("Biến thể 1 OK")

Biến thể 1 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. `Counter` | Đếm lại | `O(nq)` | Kiểm chứng |
| 2. Chia khối tiền tính | Lõi + ứng viên | `O(n√n + q√n log n)` | Truy vấn online |
| 3. Mo rollback | Chỉ thêm + hoàn tác | `O(n√q)` | **Lời giải chuẩn** (offline) |
| 4. Đếm nén | Hằng số nhỏ | `O(nq)` | `n·q` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [4,4,1,1,1]`, truy vấn `[0,4,3]`, `[0,3,2]`.

**Bài 2 (tính tay).** Chạy Mo rollback (`B = 2`) cho `nums = [1,2,1,2,2,1]`, truy vấn `[[0,5,3],[1,4,2]]`.

**Bài 3 (điền TODO).** Viết `range_max_freq(nums, queries)` chỉ trả **tần suất** lớn nhất trong mỗi đoạn.

**Bài 4 (tự viết lại).** Viết lại Cách 2 để trả lời **online** (mỗi truy vấn đến lần lượt).

**Bài 5 (mini challenge).** Đếm số giá trị phân biệt trong đoạn bằng Mo thường (có xoá).

In [18]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def range_max_freq(nums, queries):
    # TODO
    pass

In [19]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def distinct_mo(nums, queries):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[4,4,1,1,1]`: mode `1` (3 lần) ⇒ `1`. (2) `[4,4,1,1]`: hoà, chọn `1` (2 lần ≥ 2). (3) `[1, 1]`.

**Bài 2.** (1) Khối 0 (`bend = 2`): truy vấn `[0,5,3]`. (2) Phần phải `[2,5] = [1,2,2,1]` mode `1` (2). Thêm tạm `[0,1] = [1,2]` ⇒ `1` và `2` cùng 3 ⇒ chọn `1`. (3) Truy vấn `[1,4,2]` thuộc khối 0, phần phải `[2,4] = [1,2,2]`, thêm `[1]` = `2` ⇒ `2` (3 lần) ⇒ `[1, 2]`.

**Bài 3.** (1) Như Mo rollback nhưng chỉ giữ tần suất. (2) Bỏ tie-break. (3) So với `Counter`.

**Bài 4.** (1) Tiền tính `tab` như cũ. (2) Mỗi truy vấn xử lý ngay. (3) Không cần sắp truy vấn.

**Bài 5.** (1) Thêm: `cnt[v] += 1`, nếu thành 1 thì `distinct += 1`. (2) Xoá: ngược lại. (3) Sắp truy vấn theo `(l // B, r)`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
assert Solution().subarrayMajority([4, 4, 1, 1, 1], [[0, 4, 3], [0, 3, 2]]) == [1, 1]
assert mo_rollback([1, 2, 1, 2, 2, 1], [[0, 5, 3], [1, 4, 2]]) == [1, 2] == brute([1, 2, 1, 2, 2, 1], [[0, 5, 3], [1, 4, 2]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [21]:
# Bài 3, 4
def range_max_freq_sol(nums, queries):
    return [mode_freq for mode_freq in (max(Counter(nums[l:r + 1]).values()) for l, r in queries)]

class OnlineMode:
    def __init__(self, nums):
        self.vals = sorted(set(nums)); rk = {v: i for i, v in enumerate(self.vals)}; self.a = [rk[x] for x in nums]
        n = len(nums); m = len(self.vals); self.B = B = max(1, int(n ** 0.5)); self.nb = (n + B - 1) // B; self.tab = []
        for b in range(self.nb):
            cnt = [0] * m; row = [None] * n; bf = bv = 0
            for r in range(b * B, n):
                v = self.a[r]; cnt[v] += 1
                if cnt[v] > bf or (cnt[v] == bf and v < bv): bf, bv = cnt[v], v
                row[r] = (bf, bv)
            self.tab.append(row)
        self.pos = defaultdict(list)
        for i, v in enumerate(self.a): self.pos[v].append(i)
    def query(self, l, r, t):
        B = self.B; bs = (l + B - 1) // B
        if bs >= self.nb or bs * B > r:
            c = Counter(self.a[l:r + 1]); bf, bv = min(((cc, v) for v, cc in c.items()), key=lambda x: (-x[0], x[1]))
        else:
            bf, bv = self.tab[bs][r]
            for v in set(self.a[l:bs * B]):
                p = self.pos[v]; c = bisect_right(p, r) - bisect_left(p, l)
                if c > bf or (c == bf and v < bv): bf, bv = c, v
        return self.vals[bv] if bf >= t else -1

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 30); a = [rng.randint(1, 5) for _ in range(n)]; om = OnlineMode(a)
    for _ in range(10):
        l = rng.randrange(n); r = rng.randrange(l, n); t = rng.randint(1, r - l + 1)
        assert om.query(l, r, t) == brute(a, [[l, r, t]])[0]
print("Bài 3, 4 OK:", range_max_freq_sol([1, 1, 2, 2, 1, 1], [[0, 5], [2, 3]]))

Bài 3, 4 OK: [4, 2]


In [22]:
# Bài 5: số giá trị phân biệt bằng Mo thường
def distinct_mo_sol(nums, queries):
    n = len(nums); B = max(1, int(n / max(1, len(queries)) ** 0.5))
    order = sorted(range(len(queries)), key=lambda i: (queries[i][0] // B, queries[i][1] if (queries[i][0] // B) % 2 == 0 else -queries[i][1]))
    cnt = Counter(); cur = 0; cl, cr = 0, -1; out = [0] * len(queries)
    def add(i):
        nonlocal cur
        cnt[nums[i]] += 1
        if cnt[nums[i]] == 1: cur += 1
    def remove(i):
        nonlocal cur
        cnt[nums[i]] -= 1
        if cnt[nums[i]] == 0: cur -= 1
    for qi in order:
        l, r = queries[qi]
        while cr < r: cr += 1; add(cr)
        while cl > l: cl -= 1; add(cl)
        while cr > r: remove(cr); cr -= 1
        while cl < l: remove(cl); cl += 1
        out[qi] = cur
    return out

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 30); a = [rng.randint(1, 6) for _ in range(n)]; qs = []
    for _ in range(10):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    assert distinct_mo_sol(a, qs) == [len(set(a[l:r + 1])) for l, r in qs]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nén giá trị (thứ tự tăng); B = n / sqrt(q)
nhóm truy vấn theo l // B, sắp theo r
mỗi nhóm: cr = cuối khối; (bf, bv) = mode phần phải
  truy vấn gọn trong khối -> đếm trực tiếp
  ngược lại: mở rộng cr tới r (chỉ thêm); thêm tạm [l, cuối khối]; đọc mode; trừ lại
  xoá cnt sau mỗi nhóm
ans = vals[bv] nếu bf >= threshold, ngược lại -1
```

**Checklist:**

- [ ] Chọn tần suất **cao nhất** (không phải phần tử đầu đạt ngưỡng)?
- [ ] Tie-break giá trị **nhỏ nhất**?
- [ ] Hoàn tác phần trái sau mỗi truy vấn?
- [ ] Đặt lại `cnt` sau mỗi nhóm khối?
- [ ] Đã so với `Counter` trực tiếp?